# 01 — Scraping (Obtain)

Scrapes the general classification winners table for all three Grand Tours from Wikipedia.

**Sources**
| Race | Page | First edition |
|---|---|---|
| Tour de France | List of TdF GC winners | 1903 |
| Giro d'Italia | List of Giro GC winners | 1909 |
| Vuelta a España | List of Vuelta GC winners | 1935 |

**Design decision — mapping columns by header, not position.**
The three tables share a structure but not their header spellings: the sponsor column appears
as `Sponsor/Team`, `Sponsor / team` and `Sponsor/team`, and the time column as `Time/Points`,
`Time / points` and (for the Vuelta, which never used points) simply `Time`.

Indexing columns by position would work only as long as all three pages keep an identical
column count. Instead the scraper normalises each header (lowercase, collapse whitespace around
slashes) and looks it up in an alias table, so one function handles all three races and tolerates
Wikipedia adding or reordering columns.

**Design decision — no cleaning here.**
Every value is stored exactly as scraped: `'2,428km (1,509mi)'`, `'94h 33′ 14″'`, `'+ 2h 59′ 21″'`.
Parsing happens in `02_cleaning.ipynb`. Keeping Obtain and Scrub separate means cleaning logic
can be rewritten and re-run without touching the network.

In [2]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.append(str(project_root))

from src.scraper import scrape_all_races, RACE_URLS
import pandas as pd

In [3]:
records = scrape_all_races()
raw = pd.DataFrame(records)
print(raw.shape)
raw.head()

tdf: 124 rows
giro: 118 rows
vuelta: 91 rows
(333, 9)


,race,year,country,cyclist,team,distance,time_points,margin,stage_wins
0,tdf,1903,France,Maurice Garin,La Française,"2,428km (1,509mi)",94h 33′ 14″,+ 2h 59′ 21″,3
1,tdf,1904,France,Henri Cornet[b],Conte,"2,428km (1,509mi)",96h 05′ 55″,+ 2h 16′ 14″,1
2,tdf,1905,France,Louis Trousselier,Peugeot–Wolber,"2,994km (1,860mi)",35,26,5
3,tdf,1906,France,René Pottier,Peugeot–Wolber,"4,637km (2,881mi)",31,8,5
4,tdf,1907,France,Lucien Petit-Breton,Peugeot–Wolber,"4,488km (2,789mi)",47,19,2


In [4]:
print(raw["race"].value_counts(), "\n")
print("Missing values per column:")
print(raw.isna().sum(), "\n")
print("Year range per race:")
print(raw.groupby("race")["year"].agg(["min", "max", "count"]))

race
tdf       124
giro      118
vuelta     91
Name: count, dtype: int64 

Missing values per column:
race            0
year            0
country         0
cyclist         0
team            0
distance        0
time_points     0
margin          0
stage_wins     22
dtype: int64 

Year range per race:
         min   max  count
race                     
giro    1909  2026    118
tdf     1903  2026    124
vuelta  1935  2025     91


In [5]:
# rows where the cyclist field doesn't look like a name
odd = raw[~raw["cyclist"].str.contains(r"[A-Za-zÀ-ž]{3}", na=False)]
print("Rows with no plausible cyclist name:", len(odd))
odd.head(20)

Rows with no plausible cyclist name: 24


,race,year,country,cyclist,team,distance,time_points,margin,stage_wins
13,tdf,1916,—,—,—,—,—,—,NaN
14,tdf,1917,—,—,—,—,—,—,NaN
15,tdf,1918,—,—,—,—,—,—,NaN
38,tdf,1941,—,—,—,—,—,—,NaN
39,tdf,1942,—,—,—,—,—,—,NaN
40,tdf,1943,—,—,—,—,—,—,NaN
41,tdf,1944,—,—,—,—,—,—,NaN
42,tdf,1945,—,—,—,—,—,—,NaN
43,tdf,1946,—,—,—,—,—,—,—
127,giro,1912,Italy,—,Atala–Dunlop,"2,443km (1,518mi)",33,10,1


In [6]:
raw.to_csv("../data/raw/grand_tours_raw.csv", index=False)
print("Saved:", raw.shape)

Saved: (333, 9)


In [7]:
# The disputed Armstrong years
print("TdF 1998-2006:")
print(raw[(raw["race"] == "tdf") & (raw["year"].astype(int).between(1998, 2006))][
    ["year", "country", "cyclist", "time_points"]].to_string(index=False))

# Footnote markers in names
print("\nNames containing footnote markers:")
print(raw[raw["cyclist"].str.contains(r"\[", na=False)]["cyclist"].tolist())

# What formats appear in time_points?
print("\nSample of time_points values that are NOT h/m/s format:")
not_time = raw[~raw["time_points"].str.contains("h", na=False)]["time_points"].unique()
print(not_time[:30])

TdF 1998-2006:
year country          cyclist time_points
1998   Italy    Marco Pantani 92h 49′ 46″
1999       —     No winner[a]           —
2000       —     No winner[a]           —
2001       —     No winner[a]           —
2002       —     No winner[a]           —
2003       —     No winner[a]           —
2004       —     No winner[a]           —
2005       —     No winner[a]           —
2006   Spain Óscar Pereiro[d] 89h 40′ 27″

Names containing footnote markers:
['Henri Cornet[b]', 'Bjarne Riis[c]', 'No winner[a]', 'No winner[a]', 'No winner[a]', 'No winner[a]', 'No winner[a]', 'No winner[a]', 'No winner[a]', 'Óscar Pereiro[d]', 'Andy Schleck#[e]', 'Michele Scarponi†[a]', 'Roberto Heras[a]']

Sample of time_points values that are NOT h/m/s format:
<StringArray>
['35', '31', '47', '36', '37', '63', '43', '49', '—', '25', '28', '50', '33']
Length: 13, dtype: str


## What the raw scrape contains

| Race | Rows | Year range | Editions actually held |
|---|---|---|---|
| Tour de France | 124 | 1903–2026 | ~112 |
| Giro d'Italia | 118 | 1909–2026 | ~109 |
| Vuelta a España | 91 | 1935–2025 | ~80 |

The surplus rows are placeholder entries for years when no race took place.

### Edge cases identified (all left for the cleaning stage)

**1. Missing data is encoded as `—` (em-dash), not as empty cells.**
`isna()` reports zero missing values across almost every column, which is misleading — 24 rows
are war-year placeholders filled with em-dashes. Any pipeline that trusted `isna()` here would
silently carry `'—'` strings into numeric columns.

**2. `time_points` holds two incompatible units.**
Early editions (TdF 1905–1912, Giro pre-1914) ranked riders by points, not elapsed time, so the
column contains both `'94h 33′ 14″'` (a duration) and `'35'` (a point total). Only the format
distinguishes them. The same applies to `margin`.

**3. Times use typographic prime characters.** `′` (U+2032) and `″` (U+2033), not `'` and `"`.
A regex written with ASCII quotes silently matches nothing.

**4. Distances carry thousands separators and dual units**: `'2,428km (1,509mi)'`.

**5. Names carry three different annotation types, with distinct meanings:**

| Marker | Example | Meaning |
|---|---|---|
| `[a]`–`[e]` | `Henri Cornet[b]` | Wikipedia footnote reference — no data value |
| `#` | `Andy Schleck#[e]` | Title awarded retroactively after original winner stripped |
| `†` | `Michele Scarponi†[a]` | Title reassigned |

Rather than stripping these as noise, the `#`/`†` markers are extracted into an
`is_reassigned` flag before cleaning the name. Óscar Pereiro, Andy Schleck, Michele Scarponi
and Roberto Heras all inherited their titles from disqualified riders — a genuine attribute
of the result that would be lost by naive normalisation.

**6. The 1912 Giro is a genuine special case** — it has a distance, team and points but no
individual winner, because that edition ran a team classification only. This is real data, not
a gap, and must not be discarded with the war years.

**7. Two different kinds of "no result", which must not be conflated:**

| Case | Cyclist field | Meaning |
|---|---|---|
| War years (1915–18, 1940–46, 1938–40 Vuelta) | `—` | No race was held |
| TdF 1999–2005 | `No winner[a]` | Race was held; result annulled (Armstrong) |

The first is an absent edition; the second is an edition with a void result. A distance and
route existed in 1999–2005 — those years are not gaps in the sport's history, only in its
record of winners. The cleaning stage encodes this as `no_race` vs `is_disputed`.

All of these are preserved as-is here. `01` obtains; `02` decides what they mean.